# Notebook 04 · VPN, Proxy & Tor Detection with IP2Location.io

**Use case:** Review a list of IP addresses for anonymization and selected security signals using the [IP2Location.io API](https://www.ip2location.io/ip2location-documentation). Build a review-friendly dataset, summarize the findings, and export CSV reports.

**Important:** Detection is **not proof of abuse**, and an IP's classification may change. VPNs and Tor have legitimate privacy uses. Use signals as *context for investigation*, not as automatic evidence of malicious behavior.

This notebook has two modes:

- **`demo`** *(default)*: no API calls; clearly labeled **synthetic** Security-plan-style responses, using RFC 5737 documentation IP addresses so that VPN/Tor examples can be explored safely.
- **`live`**: reads `sample_security_ips.csv` (or your own CSV), validates IP addresses, and queries the actual API once per unique public IP. Does **not** claim any listed real address is currently a proxy.

> **Plan coverage (as documented in October 2026):** Free/Starter offer basic open-proxy (`PUB`) detection with `is_proxy`. Plus adds VPN and Tor detection in the `is_proxy` coverage. The Security plan adds the detailed `proxy` object (including `is_vpn`/`is_tor` and proxy type) and `fraud_score`. If detailed fields are absent, VPN/Tor classification must be **Not assessed**, even when `is_proxy` is `false`.

References: [API documentation](https://www.ip2location.io/ip2location-documentation) · [Plan features](https://www.ip2location.io/pricing).

## 1. Install dependencies

Run once in a new environment. The notebook uses `requests`, `pandas`, `matplotlib`, and Jupyter's built-in display support.

In [ ]:
%pip -q install requests pandas matplotlib

## 2. Imports and configuration

Leave `MODE = "demo"` for an offline walkthrough. To use your IP2Location.io account, change to `"live"` and provide the key with an environment variable, e.g. `IP2LOCATION_IO_API_KEY`. Keyless access is also supported, but **keyless/free access does not provide detailed VPN/Tor classifications**.

For live tests, run this notebook from the same folder as `sample_security_ips.csv`. If the sample CSV isn't present, the next cell writes it for you.

`MAX_UNIQUE_LOOKUPS` limits live requests even if a large CSV is supplied. The notebook does not perform bulk API queries.

In [ ]:
from pathlib import Path
from collections import Counter
import ipaddress
import os
import time

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display

MODE = "demo"  # Change to "live" to call the IP2Location.io API.
API_KEY = os.getenv("IP2LOCATION_IO_API_KEY", "").strip()  # Never paste a secret into shared notebooks.
API_URL = "https://api.ip2location.io/"
INPUT_CSV = Path("sample_security_ips.csv")
OUTPUT_CSV = Path("04_ip2locationio_security_review.csv")
SUMMARY_CSV = Path("04_ip2locationio_security_summary.csv")
MAX_UNIQUE_LOOKUPS = 10
REQUEST_DELAY_SECONDS = 0.25
TIMEOUT_SECONDS = 15

assert MODE in {"demo", "live"}
print(f"Mode: {MODE} | API key configured: {bool(API_KEY)}")

## 3. Load sample input and validate addresses

The live CSV has columns `case_id`, `label`, and `ip_address`. It intentionally includes a duplicate, a private address, a documentation-only address, and a malformed address. These are handled without wasting API queries.

**Privacy note:** An IP address can be personal data depending on the context. Review applicable retention and data-processing requirements before sending customer traffic to an external service.

In [ ]:
sample_rows = [
    ("web-001", "Google Public DNS", "8.8.8.8"),
    ("web-002", "Cloudflare DNS", "1.1.1.1"),
    ("web-003", "Quad9 DNS", "9.9.9.9"),
    ("web-004", "OpenDNS", "208.67.222.222"),
    ("web-005", "Google IPv6 DNS", "2001:4860:4860::8888"),
    ("web-006", "Cloudflare IPv6 DNS", "2606:4700:4700::1111"),
    ("web-007", "Duplicate Google DNS", "8.8.8.8"),
    ("web-008", "Private network (skip)", "192.168.1.20"),
    ("web-009", "Documentation address (skip)", "203.0.113.50"),
    ("web-010", "Invalid IP (skip)", "999.999.999.999"),
]
if not INPUT_CSV.exists():
    pd.DataFrame(sample_rows, columns=["case_id", "label", "ip_address"]).to_csv(INPUT_CSV, index=False)
    print(f"Created {INPUT_CSV}")

source_df = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)
required = {"case_id", "label", "ip_address"}
if not required.issubset(source_df.columns):
    raise ValueError(f"CSV requires {sorted(required)}; found {list(source_df.columns)}")

def inspect_address(value):
    try:
        parsed = ipaddress.ip_address(str(value).strip())
    except ValueError:
        return (None, "invalid_ip")
    return (str(parsed), "public" if parsed.is_global else "non_public_ip")

source_df[["normalized_ip", "ip_validation"]] = pd.DataFrame(
    source_df["ip_address"].map(inspect_address).tolist(),
    index=source_df.index,
)
display(source_df[["case_id", "label", "ip_address", "ip_validation"]])
print("Distinct public IPs in live input:", source_df.loc[source_df.ip_validation == "public", "normalized_ip"].nunique())

## 4. API lookup helper (for live mode)

Uses an HTTPS GET with `ip` and `format=json` parameters; an API key, if set, is sent in an **Authorization: Bearer** header. The response is checked for both HTTP and API-level errors. A rate-limit error stops additional lookups. We keep a per-run cache by unique normalized IP and never infer missing paid fields.

In [ ]:
def lookup_one(ip, session):
    headers = {"Accept": "application/json"}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"
    try:
        response = session.get(
            API_URL,
            params={"ip": ip, "format": "json"},
            headers=headers,
            timeout=TIMEOUT_SECONDS,
        )
        if response.status_code == 429:
            return {"lookup_status": "rate_limited", "lookup_error": "API rate limit reached", "api_data": None}
        if not response.ok:
            return {"lookup_status": "api_error", "lookup_error": f"HTTP {response.status_code}: {response.text[:160]}", "api_data": None}
        try:
            data = response.json()
        except ValueError:
            return {"lookup_status": "api_error", "lookup_error": "Non-JSON API response", "api_data": None}
        if not isinstance(data, dict):
            return {"lookup_status": "api_error", "lookup_error": "Unexpected API response type", "api_data": None}
        if "error" in data:
            return {"lookup_status": "api_error", "lookup_error": str(data["error"])[:160], "api_data": None}
        if "ip" not in data:
            return {"lookup_status": "api_error", "lookup_error": "Missing IP in API response", "api_data": None}
        return {"lookup_status": "ok", "lookup_error": None, "api_data": data}
    except requests.RequestException as exc:
        return {"lookup_status": "network_error", "lookup_error": str(exc)[:160], "api_data": None}


def run_live_lookup(df):
    cache = {}
    distinct = df.loc[df.ip_validation == "public", "normalized_ip"].drop_duplicates().tolist()
    print(f"Querying up to {MAX_UNIQUE_LOOKUPS} of {len(distinct)} unique public IPs")
    with requests.Session() as session:
        for index, ip in enumerate(distinct):
            if index >= MAX_UNIQUE_LOOKUPS:
                cache[ip] = {"lookup_status": "skipped_cap", "lookup_error": "Safety cap reached", "api_data": None}
                continue
            record = lookup_one(ip, session)
            cache[ip] = record
            print(f"  {ip}: {record['lookup_status']}")
            if record["lookup_status"] == "rate_limited":
                for rest in distinct[index + 1:]:
                    cache[rest] = {"lookup_status": "skipped_rate_limit", "lookup_error": "Stopped after a rate limit response", "api_data": None}
                break
            time.sleep(REQUEST_DELAY_SECONDS)

    results = []
    for row in df.to_dict("records"):
        if row["ip_validation"] != "public":
            payload = {"lookup_status": row["ip_validation"], "lookup_error": "Not queried", "api_data": None}
        else:
            payload = cache[row["normalized_ip"]]
        results.append({**row, **payload, "record_source": "live_api"})
    return results

## 5. Clearly synthetic demo responses

The documentation ranges `192.0.2.0/24`, `198.51.100.0/24`, and `203.0.113.0/24` are **not real live endpoints to classify**. We attach **invented labels and invented API-like data** only in demo mode, to illustrate different output possibilities: VPN, Tor, public proxy, data center, limited coverage, and threat signals. These are **not** claims about actual addresses or actual API results.

In [ ]:
def demo_records():
    def full(ip, country, proxy_type, *, is_proxy=True, fraud_score=0, threat="-", **flags):
        keys = (
            "is_vpn", "is_tor", "is_data_center", "is_public_proxy", "is_web_proxy",
            "is_residential_proxy", "is_spammer", "is_scanner", "is_botnet", "is_bogon"
        )
        proxy = {"proxy_type": proxy_type, "provider": "-", "threat": threat, "last_seen": 1}
        proxy.update({key: bool(flags.get(key, False)) for key in keys})
        return {"ip": ip, "country_name": country, "is_proxy": is_proxy, "fraud_score": fraud_score, "proxy": proxy}

    examples = [
        ("demo-001", "Synthetic VPN case", "198.51.100.10", full("198.51.100.10", "Exampleland A", "VPN", is_vpn=True, fraud_score=12)),
        ("demo-002", "Synthetic Tor case", "198.51.100.20", full("198.51.100.20", "Exampleland B", "TOR", is_tor=True, fraud_score=28)),
        ("demo-003", "Synthetic public proxy", "203.0.113.30", full("203.0.113.30", "Exampleland C", "PUB", is_public_proxy=True, fraud_score=43)),
        ("demo-004", "Synthetic data center", "203.0.113.40", full("203.0.113.40", "Exampleland D", "DCH", is_proxy=False, is_data_center=True, fraud_score=3)),
        ("demo-005", "Synthetic basic-plan response", "192.0.2.50", {"ip": "192.0.2.50", "country_name": "Exampleland E", "is_proxy": False}),
        ("demo-006", "Synthetic threat indicator", "192.0.2.60", full("192.0.2.60", "Exampleland F", "-", is_proxy=False, fraud_score=91, threat="SCANNER", is_scanner=True)),
        ("demo-007", "Duplicate VPN case", "198.51.100.10", full("198.51.100.10", "Exampleland A", "VPN", is_vpn=True, fraud_score=12)),
    ]
    return [
        {"case_id": case_id, "label": label, "ip_address": ip, "normalized_ip": ip,
         "ip_validation": "synthetic_documentation_ip", "lookup_status": "simulated",
         "lookup_error": None, "api_data": payload, "record_source": "SIMULATED_EXAMPLE"}
        for case_id, label, ip, payload in examples
    ]

records = demo_records() if MODE == "demo" else run_live_lookup(source_df)
print(f"Rows ready for analysis: {len(records)} | Source: {MODE.upper()}")

## 6. Normalize security fields without guessing

The API always makes `is_proxy` available for the documented base response, but Security-level fields are optional depending on plan. In the table below, `None`/blank means **unavailable**, not `False`. A false `is_proxy` with no nested `proxy` data must not be labeled "no VPN" or "no Tor".

The `proxy_type` codes include `VPN`, `TOR`, `PUB`, `WEB`, `DCH`, `RES`, `CPN`, and `EPN`; an IP can also carry independent signals such as `is_scanner` or `is_botnet`. `DCH` alone is not evidence of malicious behavior.

In [ ]:
BOOL_FIELDS = [
    "is_vpn", "is_tor", "is_data_center", "is_public_proxy", "is_web_proxy",
    "is_residential_proxy", "is_consumer_privacy_network", "is_enterprise_private_network",
    "is_spammer", "is_scanner", "is_botnet", "is_bogon"
]

def as_bool_or_unknown(value):
    return value if type(value) is bool else None


def meaningful(value):
    return value if value not in (None, "", "-") else None


def normalize_one(record):
    data = record.get("api_data") or {}
    proxy = data.get("proxy") if isinstance(data.get("proxy"), dict) else {}
    out = {key: record.get(key) for key in
           ("case_id", "label", "ip_address", "normalized_ip", "ip_validation", "lookup_status", "lookup_error", "record_source")}
    out.update({
        "country_name": data.get("country_name"),
        "is_proxy": as_bool_or_unknown(data.get("is_proxy")),
        "proxy_type": meaningful(proxy.get("proxy_type")),
        "provider": meaningful(proxy.get("provider")),
        "threat": meaningful(proxy.get("threat")),
        "fraud_score": data.get("fraud_score") if isinstance(data.get("fraud_score"), (int, float)) and type(data.get("fraud_score")) is not bool else None,
        "detail_available": bool(proxy),
    })
    out.update({key: as_bool_or_unknown(proxy.get(key)) for key in BOOL_FIELDS})
    return out

results_df = pd.DataFrame([normalize_one(row) for row in records])
display(results_df[["case_id", "label", "ip_address", "lookup_status", "is_proxy", "proxy_type", "is_vpn", "is_tor", "fraud_score"]])

## 7. Derive evidence-based review labels

`vpn_status` and `tor_status` use three states: **Detected**, **Not detected in returned details**, and **Not assessed**. The latter is crucial when an account does not return the Security-plan `proxy` object.

The **review priority is a notebook teaching example**, not an official IP2Location.io classification or a fraud decision system. Its illustrative `fraud_score >= 80` threshold is configurable and should be calibrated for your own context. Do not automatically block traffic solely because a connection uses a VPN, Tor, or a data center.

In [ ]:
EXAMPLE_FRAUD_SCORE_REVIEW_THRESHOLD = 80  # Teaching heuristic, not an official cutoff.


def detection_status(row, key):
    value = row[key]
    if value is True:
        return "Detected"
    if value is False:
        return "Not detected in returned details"
    return "Not assessed"


def describe_detection(row):
    if row["lookup_status"] not in ("ok", "simulated"):
        return "No API result"
    categories = [("is_vpn", "VPN"), ("is_tor", "Tor exit node"),
                  ("is_public_proxy", "Public proxy"), ("is_web_proxy", "Web proxy"),
                  ("is_residential_proxy", "Residential proxy")]
    hits = [label for key, label in categories if row[key] is True]
    if hits:
        return ", ".join(hits)
    if row["is_proxy"] is True:
        return "Proxy indicated; subtype unavailable"
    if row["is_data_center"] is True:
        return "Hosting/data center (not necessarily a proxy)"
    if row["detail_available"]:
        return "No selected proxy category detected in returned details"
    if row["is_proxy"] is False:
        return "No proxy flagged; subtype coverage unknown"
    return "Not assessed"


def review_priority(row):
    if row["lookup_status"] not in ("ok", "simulated"):
        return "Not assessed"
    threat_keys = ("is_spammer", "is_scanner", "is_botnet", "is_bogon")
    reported_threat = bool(row["threat"])
    high_score = pd.notna(row["fraud_score"]) and row["fraud_score"] >= EXAMPLE_FRAUD_SCORE_REVIEW_THRESHOLD
    if any(row[key] is True for key in threat_keys) or reported_threat or high_score:
        return "Investigate threat signal"
    if row["is_proxy"] is True or any(row[key] is True for key in ("is_vpn", "is_tor", "is_public_proxy", "is_web_proxy", "is_residential_proxy")):
        return "Review context"
    if not row["detail_available"]:
        return "Limited detection coverage"
    return "No selected threat signal"

results_df["vpn_status"] = results_df.apply(lambda row: detection_status(row, "is_vpn") if row["lookup_status"] in ("ok", "simulated") else "Not assessed", axis=1)
results_df["tor_status"] = results_df.apply(lambda row: detection_status(row, "is_tor") if row["lookup_status"] in ("ok", "simulated") else "Not assessed", axis=1)
results_df["detection_summary"] = results_df.apply(describe_detection, axis=1)
results_df["review_priority"] = results_df.apply(review_priority, axis=1)

display(results_df[["ip_address", "detection_summary", "vpn_status", "tor_status", "threat", "review_priority"]])

## 8. Summarize findings

Counts represent input rows (including duplicates) unless specified otherwise. Review coverage and errors alongside positive findings; a high number of unknowns can indicate a plan limitation rather than a clean IP list.

In [ ]:
print("Source:", "SYNTHETIC DEMONSTRATION" if MODE == "demo" else "LIVE IP2LOCATION.IO API")
print("Input rows:", len(results_df))
print("Unique input IPs:", results_df["ip_address"].nunique())
print("Lookup statuses:")
display(results_df["lookup_status"].value_counts(dropna=False).rename_axis("status").reset_index(name="rows"))
print("Review priority:")
summary_df = results_df.groupby("review_priority", dropna=False).agg(
    input_rows=("case_id", "size"),
    unique_ips=("ip_address", "nunique"),
).reset_index().sort_values(["input_rows", "review_priority"], ascending=[False, True])
display(summary_df)
print("VPN status:")
display(results_df["vpn_status"].value_counts().rename_axis("vpn_status").reset_index(name="rows"))
print("Tor status:")
display(results_df["tor_status"].value_counts().rename_axis("tor_status").reset_index(name="rows"))

## 9. Visualize findings

The charts show the **current mode only**. In demo mode they show **fictional scenarios**; in live mode they show actual API responses, subject to your subscription's available fields.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
results_df["review_priority"].value_counts().sort_values().plot(kind="barh", ax=axes[0])
axes[0].set_title("Review priorities (input rows)")
axes[0].set_xlabel("Rows")
axes[0].set_ylabel("")

categories = {"VPN": "is_vpn", "Tor": "is_tor", "Public proxy": "is_public_proxy", "Data center": "is_data_center", "Scanner": "is_scanner"}
counts = pd.Series({name: int(results_df[flag].map(lambda value: value is True).sum()) for name, flag in categories.items()})
counts.sort_values().plot(kind="barh", ax=axes[1])
axes[1].set_title("Positive detailed indicators")
axes[1].set_xlabel("Rows (multiple indicators may overlap)")
axes[1].set_ylabel("")
fig.suptitle("SYNTHETIC DEMONSTRATION — NOT REAL IP CLASSIFICATIONS" if MODE == "demo" else "LIVE IP2LOCATION.IO API RESULTS", fontsize=11)
plt.tight_layout()
plt.show()

## 10. Export review reports

Files are written to the current working directory. Both reports include a `record_source` field so demo findings cannot accidentally be mistaken for a real screening result. Do not mix synthetic example rows into a production enforcement pipeline.

In [ ]:
results_df.to_csv(OUTPUT_CSV, index=False)
summary_df.insert(0, "record_source", "SIMULATED_EXAMPLE" if MODE == "demo" else "live_api")
summary_df.to_csv(SUMMARY_CSV, index=False)
print(f"Saved {OUTPUT_CSV} ({len(results_df)} rows)")
print(f"Saved {SUMMARY_CSV} ({len(summary_df)} rows)")

## 11. How to interpret and extend this example

1. **Free/Starter:** `is_proxy` only detects categories available to the current plan (not comprehensive VPN/Tor); absent nested flags must remain **Not assessed**.
2. **Plus:** VPN/Tor contribute to detection coverage, but the absence of a detailed `proxy` object means this notebook cannot label a particular hit **VPN** or **Tor** individually. It can still show `is_proxy`.
3. **Security:** the detailed `proxy` flags, threat signals, and optional fraud score enable more specific labels. False means *not detected by the returned data at query time*, never a security guarantee.
4. **Duplicates & quotas:** this example deduplicates public IPs per notebook run; it is not the bulk API and it makes no repeated retry loops.
5. **Real decisions:** combine these indicators with account history, transaction behavior, authentication events, and human review. A VPN or Tor connection alone is not evidence of wrongdoing.

**To go live:** configure `IP2LOCATION_IO_API_KEY` in the Jupyter kernel environment (recommended: Security plan for this notebook's full set of fields); set `MODE = "live"`; replace the input CSV with your own addresses if needed; then **Run All**. The example addresses supplied for live mode are public resolvers, not presumed VPN/Tor IPs.

Documentation: https://www.ip2location.io/ip2location-documentation  
Subscription comparison: https://www.ip2location.io/pricing